# Turning data from bronze into silver

## Setup
### What the layer contains 
A different table for each faulty run in which each of the variables has been averaged for all 500 simulations and sample number has been changed to relative time from start of the scenario. All of this is duplicate to keep training and testing tests separate.
### What transformations happen
The faulty dataframes will be split for each fault, the fault free dataframes will be renamed consistently with the other tables. Values for the variables will be averaged for all simulations with the same sample and finally, a new column will be generated to transform sample into relative time
Another table will be generated in which each variable in the fault-free dataset will be analyzed for normality and parameters such as average, standard deviation and the need for normalization will be given.
At the end of the notebook, metadata will be inserted in the stats table to learn more about the variables that we're using.

## First 
I'll create the silver schema and check if all the tables are there

In [0]:
%sql
USE CATALOG tep_lakehouse;
CREATE SCHEMA IF NOT EXISTS silver;
SHOW SCHEMAS IN tep_lakehouse;

In [0]:
%sql
SHOW TABLES IN tep_lakehouse.bronze;

## Second 
I'll inspect the different tables to check column names, number of faults, etc.

In [0]:
%sql
-- Check columns and types for all 4 tables (uncomment the one you want to see)
-- DESCRIBE tep_lakehouse.bronze.tep_faultfree_training;
-- DESCRIBE tep_lakehouse.bronze.tep_faultfree_testing;
-- DESCRIBE tep_lakehouse.bronze.tep_faulty_training;
DESCRIBE tep_lakehouse.bronze.tep_faulty_testing;


In [0]:
%sql
-- Fault IDs present in faulty tables
SELECT DISTINCT faultNumber, COUNT(*) as n
FROM tep_lakehouse.bronze.tep_faulty_training
GROUP BY faultNumber
ORDER BY faultNumber;

In [0]:
%sql
-- Sample range and simulation count
SELECT 
    MIN(sample) AS min_sample,
    MAX(sample) AS max_sample,
    COUNT(DISTINCT simulationRun) AS n_simulations
FROM tep_lakehouse.bronze.tep_faulty_training;

In [0]:
%sql
-- Check nulls across all key columns in faulty_train
-- Repeat for the other 3 tables by swapping the table name
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN faultNumber   IS NULL THEN 1 ELSE 0 END) AS null_faultNumber,
    SUM(CASE WHEN simulationRun IS NULL THEN 1 ELSE 0 END) AS null_simulationRun,
    SUM(CASE WHEN sample        IS NULL THEN 1 ELSE 0 END) AS null_sample,
    SUM(CASE WHEN xmeas_1       IS NULL THEN 1 ELSE 0 END) AS null_xmeas_1,
    SUM(CASE WHEN xmv_1         IS NULL THEN 1 ELSE 0 END) AS null_xmv_1
FROM tep_lakehouse.bronze.tep_faulty_training;

In [0]:
%sql
-- Check for duplicate keys in faulty_train
SELECT
    faultNumber,
    simulationRun,
    sample,
    COUNT(*) AS n
FROM tep_lakehouse.bronze.tep_faulty_training
GROUP BY faultNumber, simulationRun, sample
HAVING COUNT(*) > 1
ORDER BY n DESC;

## Third
I'll define the constants I'll use in the cleaning of the data

In [0]:
META_COLS = ["faultNumber", "simulationRun", "sample", "source_file", "ingestion_ts"]

XMEAS_COLS = [f"xmeas_{i}" for i in range(1, 42)]
XMV_COLS   = [f"xmv_{i}"   for i in range(1, 12)]
VAR_COLS   = XMEAS_COLS + XMV_COLS  # 52 variables

FAULT_IDS  = list(range(1, 21))

# Sampling interval (same for both splits)
SAMPLE_INTERVAL_MIN = 3

# Sample ranges
SAMPLE_RANGE = {
    "train": (1, 500),   # 25 hours
    "test":  (1, 960),   # 48 hours
}

# Fault introduction point (in samples from start)
# train: fault at 1h  = 60 min  / 3 = sample 20
# test:  fault at 8h  = 480 min / 3 = sample 160
FAULT_START_SAMPLE = {
    "training": 20,
    "testing":  160,
}

# relative_time_h = (sample - 1) * 3 / 60

## Fourth
First I'll build the faulty tables by spliting the faulty tables for each fault type, then the two faultfree tables

In [0]:
from pyspark.sql import functions as F

for split in ["training", "testing"]:
    df_bronze = spark.table(f"tep_lakehouse.bronze.tep_faulty_{split}")
    # Decision: dedup on (faultNumber, simulationRun, sample) — bronze audit
    # confirmed no duplicates exist, but dropDuplicates is kept as a defensive
    # guard in case bronze is reingested with overlapping files in future
    df_bronze = df_bronze.dropDuplicates(["faultNumber", "simulationRun", "sample"])
    for fault_id in FAULT_IDS:
        # Filter to this fault only
        df_fault = df_bronze.filter(F.col("faultNumber") == fault_id)
        
        # Average all 52 variables across simulations, per sample.
        # Decision: nulls in variable columns are ignored by F.avg() which computes
        # the mean over non-null values only. Bronze audit on 5,000,000 rows confirmed
        # zero nulls in all variable columns, so no imputation or flagging is needed.
        agg_exprs = [F.avg(c).alias(c) for c in VAR_COLS]
        df_avg = df_fault.groupBy("sample").agg(*agg_exprs)
        
        # Add time columns, drop sample
        df_silver = (
            df_avg
            .withColumn("relative_time_h",   (F.col("sample") - 1) * SAMPLE_INTERVAL_MIN / 60)
            .withColumn("time_since_fault_h", (F.col("sample") - FAULT_START_SAMPLE[split]) * SAMPLE_INTERVAL_MIN / 60)
            .select("relative_time_h", "time_since_fault_h", *VAR_COLS)
            .orderBy("relative_time_h")
        )
        
        # Write to silver
        table_name = f"tep_lakehouse.silver.faulty_{split}_fault_{fault_id:02d}"
        df_silver.write.mode("overwrite").saveAsTable(table_name)
        print(f"Written: {table_name}")

In [0]:
for split in ["training", "testing"]:
    df_bronze = spark.table(f"tep_lakehouse.bronze.tep_faultfree_{split}")
    # Decision: dedup on (simulationRun, sample) — no faultNumber in fault-free tables
    df_bronze = df_bronze.dropDuplicates(["simulationRun", "sample"])

    # Average all 52 variables across simulations, per sample
    # Decision: same null handling as faulty tables — bronze confirmed zero nulls
    agg_exprs = [F.avg(c).alias(c) for c in VAR_COLS]
    df_avg = df_bronze.groupBy("sample").agg(*agg_exprs)
    
    # Add time column, drop sample
    df_silver = (
        df_avg
        .withColumn("relative_time_h", (F.col("sample") - 1) * SAMPLE_INTERVAL_MIN / 60)
        .select("relative_time_h", *VAR_COLS)
        .orderBy("relative_time_h")
    )
    
    table_name = f"tep_lakehouse.silver.faultfree_{split}"
    df_silver.write.mode("overwrite").saveAsTable(table_name)
    print(f"Written: {table_name}")

## Fifth
In this table I'll put together statistics on the different variables

In [0]:
from scipy import stats
import pandas as pd

# I Use fault-free training as the baseline — training data since that's what I'm 
# going to characterize the process against
df_ff = spark.table("tep_lakehouse.silver.faultfree_training").toPandas()

records = []
for var in VAR_COLS:
    col_data = df_ff[var].dropna()
    
    shapiro_w, shapiro_p = stats.shapiro(col_data)
    
    records.append({
        "variable":           var,
        "mean":               col_data.mean(),
        "std":                col_data.std(),
        "skewness":           stats.skew(col_data),
        "kurtosis":           stats.kurtosis(col_data),  # excess kurtosis, normal = 0
        "shapiro_w":          shapiro_w,
        "shapiro_p":          shapiro_p,
        "is_normal":          shapiro_p >= 0.05,
        "needs_normalization": shapiro_p < 0.05,
    })

df_stats = spark.createDataFrame(pd.DataFrame(records))
df_stats.write.mode("overwrite").saveAsTable("tep_lakehouse.silver.faultfree_normality_stats")
print("Written: tep_lakehouse.silver.faultfree_normality_stats")

In [0]:
%sql
-- Checking the normality and dispersion of the different variables
SELECT *
FROM tep_lakehouse.silver.faultfree_normality_stats
ORDER BY variable;

In [0]:
import matplotlib.pyplot as plt
from scipy import stats

def plot_variable_histogram(variable: str):
    # Pull data
    df = spark.table("tep_lakehouse.silver.faultfree_training").select(variable).toPandas()
    data = df[variable].dropna()
    
    # Pull normality stats for annotation
    stats_row = (
        spark.table("tep_lakehouse.silver.faultfree_normality_stats")
        .filter(f"variable = '{variable}'")
        .toPandas()
        .iloc[0]
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    
    ax.hist(data, bins=30, color="#4C72B0", edgecolor="white", linewidth=0.5)
    
    ax.axvline(stats_row["mean"], color="#C44E52", linewidth=1.5, linestyle="--", label=f'Mean: {stats_row["mean"]:.3f}')
    ax.axvline(stats_row["mean"] + stats_row["std"], color="#8C8C8C", linewidth=1, linestyle=":", label=f'±1 SD: {stats_row["std"]:.3f}')
    ax.axvline(stats_row["mean"] - stats_row["std"], color="#8C8C8C", linewidth=1, linestyle=":")

    normal_label = "Normal" if stats_row["is_normal"] else "Non-normal"
    ax.set_title(f"{variable}  |  {normal_label}  (Shapiro p = {stats_row['shapiro_p']:.4f})", fontsize=12)
    ax.set_xlabel(variable)
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()



In [0]:
# Usage
plot_variable_histogram("xmeas_13")

## Sixth
I also want to try to include "merge into" in the code but due to the nature of the data, this is not data that gets updated so I'll pretend that the testing data is an update of the training dataset

In [0]:
from delta.tables import DeltaTable

# Recompute stats from faultfree_test into a staging DataFrame
df_ff_t = spark.table("tep_lakehouse.silver.faultfree_testing").toPandas()

# Write that to a temp table (silver.faultfree_normality_stats_staging)
records = []
for var in VAR_COLS:
    col_data = df_ff_t[var].dropna()
    
    shapiro_w, shapiro_p = stats.shapiro(col_data)
    
    records.append({
        "variable":           var,
        "mean":               col_data.mean(),
        "std":                col_data.std(),
        "skewness":           stats.skew(col_data),
        "kurtosis":           stats.kurtosis(col_data),  # excess kurtosis, normal = 0
        "shapiro_w":          shapiro_w,
        "shapiro_p":          shapiro_p,
        "is_normal":          shapiro_p >= 0.05,
        "needs_normalization": shapiro_p < 0.05,
    })

df_stats = spark.createDataFrame(pd.DataFrame(records))

target = DeltaTable.forName(spark, "tep_lakehouse.silver.faultfree_normality_stats")

(target.alias("target")
    .merge(df_stats.alias("source"), "source.variable = target.variable")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
# MERGE INTO silver.faultfree_normality_stats USING the staging table ON variable
#Test it by running twice with identical data, then manually tweaking one stat in staging to verify UPDATE fires

In [0]:
%sql
DESCRIBE HISTORY tep_lakehouse.silver.faultfree_normality_stats;

## Seventh
A metadata table is joined to the stats table

In [0]:
variable_reference = [
    # XMEAS — Continuous process measurements
    {"variable": "xmeas_1",  "description": "A Feed",                              "unit": "kscmh",    "type": "XMEAS", "category": "flow",        "location": "stream_1"},
    {"variable": "xmeas_2",  "description": "D Feed",                              "unit": "kg/hr",    "type": "XMEAS", "category": "flow",        "location": "stream_2"},
    {"variable": "xmeas_3",  "description": "E Feed",                              "unit": "kg/hr",    "type": "XMEAS", "category": "flow",        "location": "stream_3"},
    {"variable": "xmeas_4",  "description": "A and C Feed",                        "unit": "kscmh",    "type": "XMEAS", "category": "flow",        "location": "stream_4"},
    {"variable": "xmeas_5",  "description": "Recycle Flow",                        "unit": "kscmh",    "type": "XMEAS", "category": "flow",        "location": "stream_8"},
    {"variable": "xmeas_6",  "description": "Reactor Feed Rate",                   "unit": "kscmh",    "type": "XMEAS", "category": "flow",        "location": "stream_6"},
    {"variable": "xmeas_7",  "description": "Reactor Pressure",                    "unit": "kPa",      "type": "XMEAS", "category": "pressure",    "location": "reactor"},
    {"variable": "xmeas_8",  "description": "Reactor Level",                       "unit": "%",        "type": "XMEAS", "category": "level",       "location": "reactor"},
    {"variable": "xmeas_9",  "description": "Reactor Temperature",                 "unit": "°C",       "type": "XMEAS", "category": "temperature", "location": "reactor"},
    {"variable": "xmeas_10", "description": "Purge Rate",                          "unit": "kscmh",    "type": "XMEAS", "category": "flow",        "location": "stream_9"},
    {"variable": "xmeas_11", "description": "Product Separator Temperature",       "unit": "°C",       "type": "XMEAS", "category": "temperature", "location": "separator"},
    {"variable": "xmeas_12", "description": "Product Separator Level",             "unit": "%",        "type": "XMEAS", "category": "level",       "location": "separator"},
    {"variable": "xmeas_13", "description": "Product Separator Pressure",          "unit": "kPa",      "type": "XMEAS", "category": "pressure",    "location": "separator"},
    {"variable": "xmeas_14", "description": "Product Separator Underflow",         "unit": "m3/hr",    "type": "XMEAS", "category": "flow",        "location": "stream_10"},
    {"variable": "xmeas_15", "description": "Stripper Level",                      "unit": "%",        "type": "XMEAS", "category": "level",       "location": "stripper"},
    {"variable": "xmeas_16", "description": "Stripper Pressure",                   "unit": "kPa",      "type": "XMEAS", "category": "pressure",    "location": "stripper"},
    {"variable": "xmeas_17", "description": "Stripper Underflow",                  "unit": "m3/hr",    "type": "XMEAS", "category": "flow",        "location": "stream_11"},
    {"variable": "xmeas_18", "description": "Stripper Temperature",                "unit": "°C",       "type": "XMEAS", "category": "temperature", "location": "stripper"},
    {"variable": "xmeas_19", "description": "Stripper Steam Flow",                 "unit": "kg/hr",    "type": "XMEAS", "category": "flow",        "location": "stripper"},
    {"variable": "xmeas_20", "description": "Compressor Work",                     "unit": "kW",       "type": "XMEAS", "category": "power",       "location": "compressor"},
    {"variable": "xmeas_21", "description": "Reactor Cooling Water Outlet Temp",   "unit": "°C",       "type": "XMEAS", "category": "temperature", "location": "reactor"},
    {"variable": "xmeas_22", "description": "Separator Cooling Water Outlet Temp", "unit": "°C",       "type": "XMEAS", "category": "temperature", "location": "separator"},

    # XMEAS 23-41 — Composition measurements (all mol%)
    # Reactor feed (stream 6) — components A–F
    {"variable": "xmeas_23", "description": "Reactor Feed Component A",            "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_6"},
    {"variable": "xmeas_24", "description": "Reactor Feed Component B",            "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_6"},
    {"variable": "xmeas_25", "description": "Reactor Feed Component C",            "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_6"},
    {"variable": "xmeas_26", "description": "Reactor Feed Component D",            "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_6"},
    {"variable": "xmeas_27", "description": "Reactor Feed Component E",            "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_6"},
    {"variable": "xmeas_28", "description": "Reactor Feed Component F",            "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_6"},
    # Purge gas (stream 9) — components A–H
    {"variable": "xmeas_29", "description": "Purge Gas Component A",               "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_9"},
    {"variable": "xmeas_30", "description": "Purge Gas Component B",               "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_9"},
    {"variable": "xmeas_31", "description": "Purge Gas Component C",               "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_9"},
    {"variable": "xmeas_32", "description": "Purge Gas Component D",               "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_9"},
    {"variable": "xmeas_33", "description": "Purge Gas Component E",               "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_9"},
    {"variable": "xmeas_34", "description": "Purge Gas Component F",               "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_9"},
    {"variable": "xmeas_35", "description": "Purge Gas Component G",               "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_9"},
    {"variable": "xmeas_36", "description": "Purge Gas Component H",               "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_9"},
    # Product (stream 11) — components D–H
    {"variable": "xmeas_37", "description": "Product Component D",                 "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_11"},
    {"variable": "xmeas_38", "description": "Product Component E",                 "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_11"},
    {"variable": "xmeas_39", "description": "Product Component F",                 "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_11"},
    {"variable": "xmeas_40", "description": "Product Component G",                 "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_11"},
    {"variable": "xmeas_41", "description": "Product Component H",                 "unit": "mol%",     "type": "XMEAS", "category": "composition", "location": "stream_11"},

    # XMV — Manipulated variables (all valve positions, all %)
    {"variable": "xmv_1",   "description": "D Feed Flow Valve",                    "unit": "%",        "type": "XMV",   "category": "valve",       "location": "stream_2"},
    {"variable": "xmv_2",   "description": "E Feed Flow Valve",                    "unit": "%",        "type": "XMV",   "category": "valve",       "location": "stream_3"},
    {"variable": "xmv_3",   "description": "A Feed Flow Valve",                    "unit": "%",        "type": "XMV",   "category": "valve",       "location": "stream_1"},
    {"variable": "xmv_4",   "description": "A and C Feed Flow Valve",              "unit": "%",        "type": "XMV",   "category": "valve",       "location": "stream_4"},
    {"variable": "xmv_5",   "description": "Compressor Recycle Valve",             "unit": "%",        "type": "XMV",   "category": "valve",       "location": "compressor"},
    {"variable": "xmv_6",   "description": "Purge Valve",                          "unit": "%",        "type": "XMV",   "category": "valve",       "location": "stream_9"},
    {"variable": "xmv_7",   "description": "Separator Pot Liquid Flow Valve",      "unit": "%",        "type": "XMV",   "category": "valve",       "location": "stream_10"},
    {"variable": "xmv_8",   "description": "Stripper Liquid Product Flow Valve",   "unit": "%",        "type": "XMV",   "category": "valve",       "location": "stream_11"},
    {"variable": "xmv_9",   "description": "Stripper Steam Valve",                 "unit": "%",        "type": "XMV",   "category": "valve",       "location": "stripper"},
    {"variable": "xmv_10",  "description": "Reactor Cooling Water Flow Valve",     "unit": "%",        "type": "XMV",   "category": "valve",       "location": "reactor"},
    {"variable": "xmv_11",  "description": "Condenser Cooling Water Flow Valve",   "unit": "%",        "type": "XMV",   "category": "valve",       "location": "separator"},
]


In [0]:
# Write to silver as a dimension table
import pandas as pd

df_var_ref = spark.createDataFrame(pd.DataFrame(variable_reference))
df_var_ref.write.mode("overwrite").saveAsTable("tep_lakehouse.silver.variable_reference")
print(f"Written: tep_lakehouse.silver.variable_reference ({len(variable_reference)} rows)")

In [0]:
df_stats    = spark.table("tep_lakehouse.silver.faultfree_normality_stats")
df_ref      = spark.table("tep_lakehouse.silver.variable_reference")

df_enriched = df_stats.join(df_ref, on="variable", how="left")

df_enriched.write.mode("overwrite").saveAsTable("tep_lakehouse.silver.faultfree_normality_stats_enriched")

In [0]:
%sql
SELECT * from tep_lakehouse.silver.faultfree_normality_stats_enriched